In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

In [ ]:
#get dataframe from railway database
import psycopg2

import os
import numpy as np

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    sslmode="require"
)

cur = conn.cursor()

# test query
#cur.execute("SELECT * FROM planet_history LIMIT 2000;")
cur.execute("""
    SELECT *
    FROM planet_history
    ORDER BY timestamp DESC;
""")

rows = cur.fetchall()

cur.close()
conn.close()



In [ ]:
#turn data into dataframe
import pandas as pd

cols = [desc[0] for desc in cur.description]

df = pd.DataFrame(rows, columns=cols)

df.head()

In [ ]:
df.columns
df = df.sort_values("timestamp")  # optional: chronological

In [ ]:
df['enemy_attacking_owner'].value_counts()

In [ ]:
df['currentOwner'].value_counts()

In [ ]:
df['in_major_order'].value_counts()

In [ ]:
#explore to make sure nothing is wrong with data
print(df[df["currentOwner"] == "Illuminate"][["name", "planet_index"]])

In [ ]:
df[df['in_major_order'] == 'T'][['name', 'currentOwner', 'in_major_order', 'timestamp']].head(20)
#df[df['name'] == 'CURIA'][['name', 'in_major_order', 'timestamp']].tail(20)

In [ ]:
print(df.columns.tolist())

In [ ]:
df['major_order_id'].tail(20)
df["major_order_id"] = df["major_order_id"].fillna(-1)
df["major_order_id"] = df["major_order_id"].astype("int64")
df.sort_values("timestamp").tail(20)["major_order_id"]


In [ ]:
#below is used to cross reference total players against other sites like https://hd2galaxy.com to make sure data is near-correct
latest_time = df['timestamp'].max()

total_players_now = df.loc[
    df['timestamp'] == latest_time,
    'player_on_planet'
].sum()

total_players_now

In [ ]:
df["name"].unique()

In [ ]:
#lets get the number of unique days, so we can figure out how many days we have collected data
num_days = df["timestamp"].dt.date.nunique()
print(f"{num_days} days")

In [ ]:
#now lets get the timestamp for a the very first day and on y-axis lets put the total_players active and then lets do the hour on the x-axis
df["date"] = df["timestamp"].dt.date

hours_per_day = df.groupby("date")["timestamp"].apply(lambda x: x.dt.hour.nunique())
hours_per_day

full_days = hours_per_day[hours_per_day >= 24]
first_full_day = full_days.index[0]

print(first_full_day)

df_full_day = df[df["date"] == first_full_day].copy()
df_full_day["hour"] = df_full_day["timestamp"].dt.hour

hourly_players = (
    df_full_day
    .groupby("hour")["player_on_planet"]
    .sum()
    .reset_index()
)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(hourly_players["hour"], hourly_players["player_on_planet"])
plt.xlabel("Hour of Day")
plt.ylabel("Total Players")
plt.title(f"Total Players by Hour (First FULL Day: {first_full_day})")
plt.show()

In [ ]:
players_ts = (
    df.groupby("timestamp")["player_on_planet"]
      .sum()
      .reset_index(name="total_players")
)

players_ts["date"] = players_ts["timestamp"].dt.date
players_ts["hour"] = players_ts["timestamp"].dt.hour

hours_per_day = players_ts.groupby("date")["hour"].nunique()
full_days = hours_per_day[hours_per_day >= 24].index

players_full = players_ts[players_ts["date"].isin(full_days)].copy()

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure()

sns.lineplot(
    data=players_full,
    x="hour",
    y="total_players",
    estimator="mean",   # average across days
    errorbar=None
)
plt.xticks(range(0, 24))  # 👈 force every hour

plt.title("Average Players by Hour (All Full Days)")
plt.xlabel("Hour")
plt.ylabel("Avg Players")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ensure datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# create date + hour columns
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

hours_per_day = df.groupby("date")["hour"].nunique()

# keep only days with full 24 hours
full_days = hours_per_day[hours_per_day >= 24].index

df_full = df[df["date"].isin(full_days)].copy()
#time series of total players over time (line graph)
ts = (
    df.groupby("timestamp")["player_on_planet"]
      .sum()
      .sort_index()
)

plt.figure()
plt.plot(ts.index, ts.values)
plt.title("Total Players Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Total Players")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#major order versus non major order players 
hourly_avg = (
    df_full.groupby("hour")["player_on_planet"]
           .sum()
           .groupby(level=0)
           .mean()
)

plt.figure()
plt.plot(hourly_avg.index, hourly_avg.values)
plt.title("Average Players by Hour (All Full Days)")
plt.xlabel("Hour of Day")
plt.ylabel("Average Players")
plt.xticks(range(0, 24))
plt.show()
# we can see around 6-10pm CST iowa time that there is a peak in players, which makes sense as that is when most people are off work and school
#however around 12:00am to 4:00am there is a signficant number of players still playing. Maybe different countries with different time zones are playing at that time, or maybe there are just some really dedicated players playing at that time.




#daily total players (line graph)
daily_players = (
    df.groupby("date")["player_on_planet"]
      .sum()
      .reset_index()
)

plt.figure()
plt.plot(daily_players["date"], daily_players["player_on_planet"])
plt.title("Total Players Per Day")
plt.xlabel("Date")
plt.ylabel("Total Players")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#below we will be trying to do time series of given the day and hour what is the total number of players on right now 
players_ts = (
    df.groupby("timestamp")["player_on_planet"]
      .sum()
      .sort_index()
      .reset_index(name="total_players")
)



#feature engineering to get day of week and hour of day
players_ts["hour"] = players_ts["timestamp"].dt.hour
players_ts["dayofweek"] = players_ts["timestamp"].dt.dayofweek

# lag features (VERY IMPORTANT)
players_ts["lag_1"] = players_ts["total_players"].shift(1)
players_ts["lag_24"] = players_ts["total_players"].shift(24)

# rolling average
players_ts["rolling_24"] = players_ts["total_players"].rolling(24).mean()

# drop NA from lagging
players_ts = players_ts.dropna()

In [ ]:
#split into x and y for train and test
X = players_ts[["hour", "dayofweek", "lag_1", "lag_24", "rolling_24"]]
y = players_ts["total_players"]


In [ ]:

from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

#run the test
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error
#calculate MAE
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)


import matplotlib.pyplot as plt
#plot actual vs predicted
plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Players")
plt.show()

In [ ]:
#lets add some more feature like is the weekend

players_ts["is_weekend"] = players_ts["dayofweek"] >= 5

#lets add in_major_order feature 

df["mo_players"] = df.apply(
    lambda row: row["player_on_planet"] if row["in_major_order"] == "T" else 0,
    axis=1
)

mo_ts = (
    df.groupby("timestamp")[["mo_players", "player_on_planet"]]
      .sum()
      .reset_index()
)

mo_ts["mo_ratio"] = mo_ts["mo_players"] / mo_ts["player_on_planet"]


players_ts = players_ts.merge(mo_ts, on="timestamp", how="left")

#lets add a new feature because the release of a warbond means a lot more players will be playing 

#first lets do a column for the release date
players_ts["warbond_release"] = (
    players_ts["timestamp"].dt.date >= pd.to_datetime("2026-04-28").date()
).astype(int)

warbond_date = pd.to_datetime("2026-04-28")
#now lets add a column for hours since warbond release, which will be 0 before the release and then start increasing after the release
players_ts["hours_since_warbond"] = (
    players_ts["timestamp"] - warbond_date
).dt.total_seconds() / 3600

# keep only positive values
players_ts["hours_since_warbond"] = players_ts["hours_since_warbond"].clip(lower=0)

X = players_ts[
    ["hour", "dayofweek", "lag_1", "lag_24", "rolling_24", "mo_ratio", "hours_since_warbond"]
]

In [ ]:
#THIS IS FOR ANALYZING THE UPLIFT OF THE WARBOND RELEASE ON THAT SPECIFIC DATE, SO WILL NEED TO CHANGE 

warbond_date = pd.Timestamp("2026-04-28")

before = players_ts[
    players_ts["timestamp"] < warbond_date
]["total_players"].mean()

after = players_ts[
    (players_ts["timestamp"] >= warbond_date) &
    (players_ts["timestamp"] < warbond_date + pd.Timedelta(days=1))
]["total_players"].mean()

uplift = (after - before) / before
print(uplift)
df["warbond_uplift"] = uplift

In [ ]:
#THIS IS WHERE WE MANUALLY UPDATE THE WARBOND RELEASE DATE AND TITLE
df["warbond_title"] = np.where(
    df["timestamp"] >= pd.Timestamp("2026-04-28"),
    "EXO EXPERTS",
    None
)

In [ ]:
print(players_ts.columns)

In [ ]:
#lets try XGboost 

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)
#predict
y_pred = model.predict(X_test)

#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
#the output from above told us lags are most important, so lets use ACF first

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

series = players_ts["total_players"]

plt.figure(figsize=(12,5))
plot_acf(series, lags=48)
plt.title("ACF - Total Players")
plt.show()

plt.figure(figsize=(12,5))
plot_pacf(series, lags=48)
plt.title("PACF - Total Players")
plt.show()

In [ ]:
players_ts["hour_sin"] = np.sin(2 * np.pi * players_ts["hour"] / 24)
players_ts["hour_cos"] = np.cos(2 * np.pi * players_ts["hour"] / 24)

features = [
    "hour_sin",
    "hour_cos",
    "dayofweek",
    "is_weekend",
    "lag_1",
    "rolling_24",
    "mo_ratio"
]

print(X_train.columns)
X = players_ts[features]
y = players_ts["total_players"]
print(X_train.columns)

In [ ]:
# --- 1. SORT (CRITICAL) ---
players_ts = players_ts.sort_values("timestamp").reset_index(drop=True)

# --- 2. DEFINE FEATURES ---
features = [
    "hour_sin",
    "hour_cos",
    "dayofweek",
    "is_weekend",
    "lag_1",
    "rolling_24",
    "mo_ratio"
]

X = players_ts[features]
y = players_ts["total_players"]

# --- 3. SPLIT INDEX ---
split_idx = int(len(X) * 0.8)

# --- 4. SPLIT (TIME-ORDERED) ---
X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

# --- 5. VERIFY ---
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain time range:")
print(players_ts.iloc[:split_idx]["timestamp"].min(), "→", players_ts.iloc[:split_idx]["timestamp"].max())

print("\nTest time range:")
print(players_ts.iloc[split_idx:]["timestamp"].min(), "→", players_ts.iloc[split_idx:]["timestamp"].max())

In [ ]:
#THIS IS WHERE TO TRY NEW FEATURES, BELOW IS THE MODEL

combat_ts = (
    df.groupby("timestamp")[["missions_won", "missions_lost"]]
      .sum()
      .reset_index()
)

combat_ts["combat_activity"] = (
    combat_ts["missions_won"] + combat_ts["missions_lost"]
)

players_ts = players_ts.merge(
    combat_ts[["timestamp", "combat_activity"]],
    on="timestamp",
    how="left"
)







In [ ]:
players_ts.columns

In [ ]:
for lag in range(1, 6):
    players_ts[f"lag_{lag}"] = players_ts["total_players"].shift(lag)

lag_features = [f"lag_{lag}" for lag in range(1, 6)]
#double check to make sure the lags are being added
print([col for col in players_ts.columns if "lag_" in col])
#set timestamp as index for time series
players_ts = players_ts.set_index("timestamp")

In [ ]:



# features = [
#     "hour_sin",
#     "hour_cos",
#     "dayofweek",
#     "is_weekend",
#     "mo_ratio",
#     "combat_activity"
# ] + lag_features


#X = players_ts.drop(columns=["total_players"])
X = players_ts.drop(columns=["total_players", "player_on_planet"])
y = players_ts["total_players"]


# --- 3. SPLIT INDEX ---
split_idx = int(len(X) * 0.8)

# --- 4. SPLIT (TIME-ORDERED) ---
X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

#lets try XGboost 

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)
#predict
y_pred = model.predict(X_test)

#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure(figsize=(18, 8))
plt.plot(y_test.index, y_test, label="Actual")
plt.plot(y_test.index, y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.head(20))

In [ ]:
players_ts.index.unique()

print(players_ts.index.min())
print(players_ts.index.max())

In [ ]:
plt.figure(figsize=(16,7))

# full actual
plt.plot(players_ts.index, players_ts["total_players"], label="Actual (Full)", alpha=0.5)

# predictions only on test
plt.plot(y_test.index, y_pred, label="Predicted (Test)", color="orange")

plt.legend()
plt.title("Full Timeline with Predictions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16,7))

plt.plot(y_train.index, y_train, label="Train")
plt.plot(y_test.index, y_test, label="Test")
plt.plot(y_test.index, y_pred, label="Predicted")

plt.legend()
plt.title("Train/Test Split Visualization")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#%pip install openai

In [ ]:
#do some data cleaning before feeding to LLM

#below makes sure we aren't feeding it NONE or bad story lines that came out by accident
df = df[
    (df["major_order_dispatch"].notna()) &
    (df["major_order_dispatch"] != "NONE") &
    (~df["major_order_dispatch"].str.contains("story", case=False, na=False))
]

unique_orders = df[
    ["major_order_id", "major_order_dispatch"]
].drop_duplicates()


In [ ]:
import pandas as pd
import json
import os
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# ==============================
# LOAD ENV + CLIENT
# ==============================
load_dotenv()
client = OpenAI()

# ==============================
# CLEAN YOUR DATA FIRST
# ==============================
df = df[
    (df["major_order_dispatch"].notna()) &
    (df["major_order_dispatch"] != "NONE") &
    (~df["major_order_dispatch"].str.contains("story", case=False, na=False))
]

# ==============================
# LOAD CACHE (SO YOU DON'T RE-CALL LLM)
# ==============================
CACHE_FILE = "mo_features_cache.csv"

if os.path.exists(CACHE_FILE):
    mo_features = pd.read_csv(CACHE_FILE)
    print("Loaded cached classifications")
else:
    mo_features = pd.DataFrame()

# ==============================
# GET UNIQUE MAJOR ORDERS
# ==============================
unique_orders = df[
    ["major_order_id", "major_order_dispatch"]
].dropna().drop_duplicates()

# Remove already-classified IDs
if not mo_features.empty:
    unique_orders = unique_orders[
        ~unique_orders["major_order_id"].isin(mo_features["major_order_id"])
    ]

print("Unique orders to classify:", len(unique_orders))

# SAFETY CHECK (prevents accidental huge API usage)
if len(unique_orders) > 50:
    raise ValueError("Too many LLM calls — something is wrong")

# ==============================
# LLM CLASSIFIER FUNCTION
# ==============================
def classify_text(text):
    prompt = f"""
Classify this Helldivers 2 major order:

\"\"\"{text}\"\"\"

Return ONLY JSON with this schema:
{{
  "mo_type": "attack | defense | liberation | strategic",
  "mo_enemy": "terminids | automatons | illuminate | unknown",
  "mo_urgency": "low | medium | high",
  "mo_scale": "single | multi"
}}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": "You are a strict JSON classifier. Output ONLY valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        content = response.choices[0].message.content.strip()

        # Extract JSON safely
        start = content.find("{")
        end = content.rfind("}") + 1
        json_str = content[start:end]

        result = json.loads(json_str)

        return {
            "mo_type": result.get("mo_type"),
            "mo_enemy": result.get("mo_enemy"),
            "mo_urgency": result.get("mo_urgency"),
            "mo_scale": result.get("mo_scale"),
        }

    except Exception as e:
        print("LLM ERROR:", e)
        return {
            "mo_type": None,
            "mo_enemy": None,
            "mo_urgency": None,
            "mo_scale": None
        }

# ==============================
# BATCH CLASSIFY (ONLY NEW)
# ==============================
new_results = []

for _, row in tqdm(unique_orders.iterrows(), total=len(unique_orders)):
    mo_id = row["major_order_id"]
    text = row["major_order_dispatch"]

    result = classify_text(text)

    new_results.append({
        "major_order_id": mo_id,
        "mo_type": result["mo_type"],
        "mo_enemy": result["mo_enemy"],
        "mo_urgency": result["mo_urgency"],
        "mo_scale": result["mo_scale"]
    })

new_df = pd.DataFrame(new_results)

# ==============================
# APPEND TO CACHE + SAVE
# ==============================
if not mo_features.empty:
    mo_features = pd.concat([mo_features, new_df], ignore_index=True)
else:
    mo_features = new_df

mo_features.to_csv(CACHE_FILE, index=False)
print("Saved updated cache")

# ==============================
# MERGE BACK INTO MAIN DF
# ==============================
df = df.merge(mo_features, on="major_order_id", how="left")

# ==============================
# CLEAN OLD DUMMY COLUMNS
# ==============================
df = df.loc[:, ~df.columns.str.startswith((
    "mo_type_", "mo_enemy_", "mo_urgency_", "mo_scale_"
))]

# ==============================
# ONE-HOT ENCODE FOR ML
# ==============================
df = pd.get_dummies(
    df,
    columns=["mo_type", "mo_enemy", "mo_urgency", "mo_scale"],
    dummy_na=True
)

print("Done. LLM features added to dataframe.")

In [ ]:
pd.set_option('display.max_columns', None)
df.head()


In [ ]:
df['major_order_dispatch'].unique()